# Chapter 8: Avoiding Hallucinations

## Lesson

**Hallucinations** are confident but incorrect claims made by Claude. This is one of the most important failure modes to understand and mitigate.

### Two Primary Strategies

1. **Give Claude an out**: Explicitly tell Claude it's okay to say "I don't know" or that it should only answer with certainty
2. **Require evidence first**: Make Claude extract relevant quotes or citations from provided documents before answering

### When Hallucinations Happen
- Factual questions about obscure topics
- Questions requiring very precise numbers or dates
- When Claude is asked to reference specific documents without being given them
- At higher temperature settings

In [ ]:
import anthropic

%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt: str = "", prefill: str = ""):
    messages = [{"role": "user", "content": prompt}]
    if prefill:
        messages.append({"role": "assistant", "content": prefill})
    kwargs = {
        "model": MODEL_NAME,
        "max_tokens": 2000,
        "temperature": 0.0,
        "messages": messages
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    message = client.messages.create(**kwargs)
    return message.content[0].text

### Strategy 1: Give Claude an Out

Without permission to say "I don't know", Claude may fabricate an answer:

In [ ]:
# Risky: Claude may hallucinate a specific weight
print("--- Without an out ---")
response = get_completion("What was the name and exact weight of the heaviest hippo ever recorded?")
print(response)
print()

# Safer: Claude can say it's not certain
print("--- With permission to say 'I don't know' ---")
response = get_completion(
    "What was the name and exact weight of the heaviest hippo ever recorded? "
    "Only answer if you know the answer with certainty. If you're not sure, say so."
)
print(response)

### Strategy 2: Require Evidence First

When working with documents, make Claude extract relevant quotes before answering. This grounds responses in actual evidence:

In [ ]:
# Simulated document
DOCUMENT = """Acme Corp Annual Report 2024

Revenue grew 15% year-over-year to $2.3 billion. Operating expenses increased 
by 8% due to expansion into three new markets. The company hired 500 new 
employees, bringing total headcount to 4,200. Customer satisfaction scores 
remained above 90% for the third consecutive year.

The board approved a $50 million investment in AI research and development.
Net income was $340 million, up from $290 million in the prior year."""

QUESTION = "What was Acme Corp's profit margin?"

# Without citation requirement — may hallucinate or calculate incorrectly
print("--- Without citation requirement ---")
response = get_completion(f"""<document>{DOCUMENT}</document>

{QUESTION}""")
print(response)
print()

# With citation requirement — grounded in evidence
print("--- With citation requirement ---")
response = get_completion(f"""<document>{DOCUMENT}</document>

{QUESTION}

First, extract the relevant quotes from the document in <quotes> tags.
Then provide your answer in <answer> tags.
If the document doesn't contain enough information to answer, say so.""")
print(response)

---
## Exercises

### Exercise 8.1
The prompt below may cause Claude to hallucinate. Fix it so Claude gives an accurate answer or admits uncertainty.

In [ ]:
# Exercise 8.1 - Fix the prompt to prevent hallucination
PROMPT = "How many studio albums has Beyonce released and what are all their names?"

# TODO: Modify the prompt to prevent hallucination
# Hint: Give Claude permission to be uncertain or qualify its answer

response = get_completion(PROMPT)
print(response)

# Grading - response should show uncertainty or qualification
def grade_exercise_8_1(response):
    hedging_words = ["not certain", "not sure", "may not", "might not", "approximate", 
                     "as of my", "knowledge cutoff", "i believe", "to the best", "i'm not"]
    resp_lower = response.lower()
    return any(word in resp_lower for word in hedging_words)

print("\n" + ("✅ PASS" if grade_exercise_8_1(response) else "❌ TRY AGAIN — Claude should express uncertainty"))

### Exercise 8.2
Using the document below, write a prompt that makes Claude answer the question using **only** information from the document, with citations.

In [ ]:
# Exercise 8.2
REPORT = """TechStart Q3 2024 Report

Monthly active users grew from 1.2 million to 1.8 million during Q3.
Premium subscribers reached 150,000, up 25% from Q2.
Average revenue per user (ARPU) was $12.50.
The company expanded to 5 new countries in Europe.
Customer churn rate decreased from 5.2% to 3.8%."""

QUESTION = "How many new premium subscribers did TechStart gain in Q3?"

# TODO: Write a prompt that requires Claude to cite evidence from the document
PROMPT = f"[Your prompt here using REPORT and QUESTION]"

response = get_completion(PROMPT)
print(response)

# Grading
def grade_exercise_8_2(response):
    return "<quotes>" in response or "quote" in response.lower() or "150,000" in response

print("\n" + ("✅ PASS" if grade_exercise_8_2(response) else "❌ TRY AGAIN — Response should include citations from the document"))

---
### Example Playground

In [ ]:
# Playground - test hallucination mitigation
PROMPT = """What is the exact population of my hometown?

If you don't have enough information to answer accurately, explain what you'd need to know."""

response = get_completion(PROMPT)
print(response)